In [1]:
import pandas as pd
import os
import glob

In [2]:
folder_path = "citi_bike"

In [3]:
print(os.listdir(folder_path))

['202207-citibike-tripdata_2.csv', '202207-citibike-tripdata_3.csv', '202207-citibike-tripdata_1.csv', '202208-citibike-tripdata_1.csv', '202210-citibike-tripdata_1.csv', '202207-citibike-tripdata_4.csv', '202203-citibike-tripdata_2.csv', '202210-citibike-tripdata_2.csv', '202203-citibike-tripdata_1.csv', '202210-citibike-tripdata_3.csv', '202204-citibike-tripdata_1.csv', '202204-citibike-tripdata_2.csv', '202204-citibike-tripdata_3.csv', '202211-citibike-tripdata_3.csv', '202202-citibike-tripdata_1.csv', '202211-citibike-tripdata_2.csv', '202209-citibike-tripdata_4.csv', '202206-citibike-tripdata_4.csv', '202211-citibike-tripdata_1.csv', '202209-citibike-tripdata_1.csv', '202206-citibike-tripdata_1.csv', '202206-citibike-tripdata_3.csv', '202209-citibike-tripdata_2.csv', '202209-citibike-tripdata_3.csv', '202206-citibike-tripdata_2.csv', '202205-citibike-tripdata_3.csv', '202205-citibike-tripdata_2.csv', '202205-citibike-tripdata_1.csv', '202212-citibike-tripdata_1.csv', '202212-citib

In [4]:
all_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

# Create an empty list and read files one by one
citibike_df = pd.DataFrame()  

for file in all_files:
    temp_df = pd.read_csv(os.path.join(folder_path, file), low_memory=False)
    citibike_df = pd.concat([citibike_df, temp_df], ignore_index=True)

# Save merged data
citibike_df.to_csv("citibike_2022.csv", index=False)

In [5]:
import requests

NOAA_TOKEN = "fWiFdeeYGQIfiICkWYPWnYTEeTuTjlfm"
url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USW00014732",  # LaGuardia Airport
    "startdate": "2022-01-01",
    "enddate": "2022-12-31",
    "datatypeid": ["TMAX", "TMIN", "PRCP"],  # Max temp, min temp, precipitation
    "limit": 1000
}


In [6]:
headers = {"token": NOAA_TOKEN}

response = requests.get(url, headers=headers, params=params)
weather_data = response.json()

In [7]:
citibike_columns = pd.read_csv("citibike_2022.csv", nrows=0).columns
weather_columns = pd.read_csv("laguardia_weather_2022.csv", nrows=0).columns

print("Citibike Columns:", list(citibike_columns))
print("Weather Columns:", list(weather_columns))

Citibike Columns: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
Weather Columns: ['date', 'datatype', 'station', 'attributes', 'value']


In [8]:
# Convert to DataFrame and save
weather_df = pd.DataFrame(weather_data["results"])
weather_df.to_csv("laguardia_weather_2022.csv", index=False)

In [9]:
# Load necessary columns from Citibike data
citibike_df = pd.read_csv("citibike_2022.csv", usecols=['ride_id', 'started_at'], low_memory=False)

# Load necessary columns from Weather data
weather_df = pd.read_csv("laguardia_weather_2022.csv", usecols=['date', 'datatype', 'value'], low_memory=False)

In [10]:
# Print Columns to Confirm
print("Citibike Columns BEFORE Processing:", citibike_df.columns)
print("Weather Columns BEFORE Processing:", weather_df.columns)

Citibike Columns BEFORE Processing: Index(['ride_id', 'started_at'], dtype='object')
Weather Columns BEFORE Processing: Index(['date', 'datatype', 'value'], dtype='object')


In [11]:
# Convert Citibike 'started_at' to date format
citibike_df["date"] = pd.to_datetime(citibike_df["started_at"], errors="coerce").dt.date

# Convert Weather 'date' column to date format
weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.date

# Drop any rows where date conversion failed
citibike_df.dropna(subset=["date"], inplace=True)
weather_df.dropna(subset=["date"], inplace=True)

In [12]:
weather_df = weather_df.groupby("date")["value"].mean().reset_index()

In [13]:
merged_df = citibike_df.merge(weather_df, on="date", how="left")

In [14]:
print("Columns in Merged Data:", merged_df.columns)

Columns in Merged Data: Index(['ride_id', 'started_at', 'date', 'value'], dtype='object')


In [15]:
merged_df.to_csv("citibike_weather_2022.csv", index=False)
print("Merged file saved successfully as 'citibike_weather_2022.csv'!")

Merged file saved successfully as 'citibike_weather_2022.csv'!


In [16]:
weather_df.head()

,date,value
0,2022-01-01,144.000000
1,2022-01-02,68.333333
2,2022-01-03,-1.333333
3,2022-01-04,-12.666667
4,2022-01-05,50.000000
